# Inspect forcing

- `sf_Atl` depth-vs-year at 26N and 48N (4 members, one row each, no mean panel).
- Control-variable timeseries (10 members) in two 1x5 figures: raw and 10-year rolling mean.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "kews" / "__init__.py").is_file())
NOTEBOOK_DIR = REPO_ROOT / "experiments/amoc"
for path in (REPO_ROOT, NOTEBOOK_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
OUTPUT_DIR = REPO_ROOT / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# -------------------------
# 1) State profile for AMOC
# -------------------------

from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl
import cmocean

from amoc_utils import (
    DEFAULT_SCENARIOS,
    MEMBER_LABELS_10,
    C1_MEMBERS_DEFAULT,
    load_lat_slice_time_depth,
    preprocess_control,
    plot_grouped_on_axis,
)

# -------------------------
# Config
# -------------------------
DATA_DIR = NOTEBOOK_DIR / "data"
DATA_FILE = DATA_DIR / 'amoc_si_mldmax_2Juan.nc'

SCENARIOS_4 = ['c', 'g', 'a', 'i'] # list(DEFAULT_SCENARIOS)  # cg, ai
LAT_TARGETS = [26.0, 48.0]
YEAR_MIN, YEAR_MAX = 2015, 2400

FIELD = 'sf_Atl'
CMAP = cmocean.cm.balance
DEPTH_EXT = [0, 5000]
SF_VMIN, SF_VMAX = 0.0, 30.0

CONTROL_VARS = [
    'si_flux_denmarkstr_ssp245',
    'mldmax_labr_ssp245',
    'mldmax_irm_ssp245',
    'mldmax_sco_ssp245',
    'mldmax_gin_ssp245',
]
CONTROL_PREPROCESS = [
    ('none', 'Raw'),
    ('rolling_mean', '10-year rolling mean'),
]
ROLLING_WINDOW = 10

LABEL_FONTSIZE = 16
YLABEL_FONTSIZE = 12
LEGEND_FONTSIZE = 14
TICK_FONTSIZE = 13


# -------------------------
# 1) sf_Atl depth-vs-year at 26N and 48N (4 members, one row each)
# -------------------------
for lat_target in LAT_TARGETS:
    slices = {}
    lat_actual = {}
    depth_ref = None

    for s in SCENARIOS_4:
        years_s, depth_s, data_s, lat_s, _ = load_lat_slice_time_depth(
            DATA_DIR,
            s,
            lat_target,
            preferred_field=FIELD,
            year_min=YEAR_MIN,
            year_max=YEAR_MAX,
        )
        slices[s] = (years_s, data_s)
        lat_actual[s] = lat_s
        if depth_ref is None:
            depth_ref = depth_s

    common_years = np.array(
        sorted(set.intersection(*[set(slices[s][0].tolist()) for s in SCENARIOS_4])),
        dtype=int,
    )

    aligned = []
    for s in SCENARIOS_4:
        years_s, data_s = slices[s]
        idx_map = {int(y): i for i, y in enumerate(years_s)}
        idx = [idx_map[int(y)] for y in common_years]
        aligned.append(data_s[:, idx])
    aligned = np.stack(aligned, axis=0)  # (4, depth, time)

    norm = mpl.colors.Normalize(vmin=SF_VMIN, vmax=SF_VMAX)

    fig, axes = plt.subplots(1, len(SCENARIOS_4), figsize=(18, 4), sharey=True, constrained_layout=False)

    mappable = None
    for i, s in enumerate(SCENARIOS_4):
        ax = axes[i]
        mappable = ax.pcolormesh(common_years, depth_ref, aligned[i], cmap=CMAP, norm=norm, shading='auto')
        ax.set_ylim(DEPTH_EXT)
        ax.invert_yaxis()
        ax.set_xlabel('Year', fontsize=LABEL_FONTSIZE)
        if i == 0:
            ax.set_ylabel('Depth (m)', fontsize=YLABEL_FONTSIZE)
        ax.tick_params(labelsize=TICK_FONTSIZE)
        # ax.set_title(f'SSP245{s} @ {lat_actual[s]:.1f}°N', fontsize=LABEL_FONTSIZE)

    fig.tight_layout(rect=[0.03, 0.08, 0.97, 0.93])
    cax = fig.add_axes([0.20, 0.03, 0.60, 0.03])
    cbar = fig.colorbar(mappable, cax=cax, orientation='horizontal')
    cbar.set_label('Streamfunction [Sv]', fontsize=YLABEL_FONTSIZE)
    cbar.ax.tick_params(labelsize=TICK_FONTSIZE)
    plt.savefig(f'{OUTPUT_DIR}/amoc_str_profile_{lat_target:.0f}.png', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
# -------------------------
# 2) Control timeseries (10 members), 1x5 layout, two figures
#    - light dashed member trajectories
#    - thick C1/C0 means
# -------------------------
ds = xr.open_dataset(DATA_FILE)
years = 2015 + ds['time'].values.astype(int)

member_labels = np.array(MEMBER_LABELS_10)
c1_mask = np.array([m in C1_MEMBERS_DEFAULT for m in member_labels], dtype=bool)

for mode, mode_label in CONTROL_PREPROCESS:
    fig, axes = plt.subplots(1, len(CONTROL_VARS), figsize=(24, 4), sharex=True)

    for j, var in enumerate(CONTROL_VARS):
        data = np.asarray(ds[var].values, dtype=np.float32)  # (10, time)
        data = preprocess_control(data, mode=mode, rolling_window=ROLLING_WINDOW)

        plot_grouped_on_axis(
            axes[j],
            years,
            data,
            c1_mask,
            ylabel=var if j == 0 else '',
            title=var.replace('_ssp245', ''),
            log_scale=False,
        )

    axes[0].legend(frameon=False, fontsize=LEGEND_FONTSIZE)
    fig.suptitle(f'Control timeseries ({mode_label})', y=1.02, fontsize=LABEL_FONTSIZE)
    plt.tight_layout()
    plt.show()
